RVC 笔记本 - 语音转换与训练（完整版）

本笔记本在**笔记本内部**完成所有操作，无需启动 WebUI。

## 功能模块
- **环境配置**：克隆仓库、下载模型包、安装依赖、环境自检
- **语音转换 (VC)**：加载模型，对音频进行音色转换
- **人声分离 (MSST)**：分离人声与伴奏
- **索引训练**：训练 FAISS 索引
- **模型导出**：保存训练好的模型

## 单元执行顺序
| 单元 | 内容 |
| --- | --- |
| 帮助函数 | run / pip_install / show_tail 等小工具 |
| 参数配置 | 仓库、安装包路径等（按需修改） |
| 环境检查 | Python / GPU / 磁盘空间 |
| 系统工具 | ffmpeg、aria2、7z、py7zr 等 |
| 克隆仓库 | 克隆本地仓库 |
| 下载完整包 | 从 HuggingFace 下载模型包 |
| 校验模型 | 校验核心模型，缺失时可补齐 |
| 安装依赖 | torch cu130 + requirements |
| 环境自检 | 导入 torch |
| 语音转换 | 参数配置 + 模型加载与推理 |
| 人声分离 | 参数配置 + PyMSS 模型分离 |
| 模型训练 | 参数配置 + 一键训练 |
| 索引训练 | 参数配置 + FAISS 索引训练 |
| 模型导出 | 参数配置 + 保存训练模型 |
| 批量处理 | 参数配置 + 目录级批量转换 |

> 前置要求：Python 3.12 x64, NVIDIA CUDA 13.0, 磁盘 >= 30GB

In [ ]:
# 帮助函数
import os, sys, shutil, socket, re, time, glob, json, subprocess, pathlib, platform, hashlib, urllib.request
from pathlib import Path

def run(cmd, check=True, timeout=None):
    print(">>> " + cmd, flush=True)
    r = subprocess.run(cmd, shell=True, executable="/bin/bash", text=True,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, timeout=timeout)
    if r.stdout and r.stdout.strip():
        print(r.stdout.strip()[-6000:])
    if check and r.returncode != 0:
        raise RuntimeError("命令失败 (rc=%s): %s" % (r.returncode, cmd))
    return r

def pip_install(req, extra=""):
    cmd = '"%s" -m pip install --disable-pip-version-check -q %s %s' % (sys.executable, extra, req)
    r = run(cmd, check=False)
    if r.returncode != 0:
        print("[pip] 重试：添加 --break-system-packages")
        run(cmd + " --break-system-packages", check=False)

def show_tail(path, n=80):
    p = Path(path)
    if not p.exists():
        print("文件不存在：", path)
        return
    lines = p.read_text(errors="replace").splitlines()
    print("\n".join(lines[-n:]))

def merge_tree(src, dst):
    src, dst = Path(src), Path(dst)
    copied = 0
    total_bytes = 0
    for s in src.rglob("*"):
        if not s.is_file():
            continue
        rel = s.relative_to(src)
        d = dst / rel
        try:
            if d.exists() and d.stat().st_size == s.stat().st_size:
                continue
            d.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(str(s), str(d))
            copied += 1
            total_bytes += s.stat().st_size
        except Exception as e:
            print("[警告] 复制失败 %s -> %s: %s" % (s, d, e))
    return copied, total_bytes


In [ ]:
# 参数配置（按需修改）
REPO_URL = "https://github.com/miniworldRuman/Retrieval-based-Voice-Conversion-WebUI.git"
WORK_DIR = os.environ.get("RVC_WORK_DIR", "/root/RVC")

HF_REPO = "lj1995/VoiceConversionWebUI"
HF_ASSETS_DIR = str(Path(WORK_DIR) / "assets")
PROJECT_ROOT = Path(WORK_DIR)
VERIFY_SHA256  = False

TORCH_INDEX  = "https://download.pytorch.org/whl/cu130"
PYPI_INDEX   = ""  # 默认源
FALLBACK_DOWNLOAD = True

print("配置完成  工作目录：", WORK_DIR)
print("HuggingFace 仓库：", HF_REPO)
print("TORCH 源          ：", TORCH_INDEX)


In [ ]:
# 环境检查
print("Python 版本 ：", sys.version.split()[0], " (目标 Python 3.12)")
print("平台        ：", platform.platform(), "|", platform.machine())
if shutil.which("nvidia-smi"):
    run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", check=False)
else:
    print("[提示] 未找到 nvidia-smi，推理/训练将回退到 CPU。")
for d in ("/", "/tmp"):
    u = shutil.disk_usage(d)
    print("磁盘 %-6s 剩余 %8.1f GB" % (d, u.free / 1e9))
if Path(WORK_DIR).exists():
    print("[提示] %s 已存在，" % WORK_DIR, "克隆单元会 git pull 更新。")


In [ ]:
# 安装系统工具与 Python 库
run("apt-get update -qq >/dev/null 2>&1 || true", check=False)
run("apt-get install -y -qq git ffmpeg aria2 p7zip-full libportaudio2 libsndfile1 >/dev/null 2>&1 || true", check=False)
pip_install("py7zr huggingface_hub tqdm requests")

print("git      :", shutil.which("git"))
print("ffmpeg   :", shutil.which("ffmpeg"))
print("aria2c   :", shutil.which("aria2c"))
print("7z       :", shutil.which("7z") or shutil.which("7zz") or "(将使用 py7zr)")


In [ ]:
# 克隆仓库
if not Path(WORK_DIR).exists():
    Path(WORK_DIR).parent.mkdir(parents=True, exist_ok=True)
    run("git clone --depth 1 %s %s" % (REPO_URL, WORK_DIR))
else:
    run("git -C %s pull --ff-only" % WORK_DIR, check=False)

if shutil.which("git-lfs"):
    run("git -C %s lfs pull 2>/dev/null || true" % WORK_DIR, check=False)

os.chdir(WORK_DIR)
print("工作目录：", os.getcwd())
print("仓库内容：", ", ".join(sorted(os.listdir("."))))


In [ ]:
# 从 HuggingFace 下载完整包（各子文件夹），不下载根目录文件
from huggingface_hub import snapshot_download, hf_hub_download

ASSET_SUBFOLDERS = [
    "hubert_base",
    "pretrained",
    "pretrained_v2",
    "pymss_weights",
    "weights",
]

assets_dir = Path(HF_ASSETS_DIR)
assets_dir.mkdir(parents=True, exist_ok=True)

for folder in ASSET_SUBFOLDERS:
    dest = assets_dir / folder
    if dest.exists() and any(dest.iterdir()):
        print("已存在，跳过：", dest)
        continue
    print("下载 %s ..." % folder)
    tmp = snapshot_download(repo_id=HF_REPO, repo_type="model", allow_patterns="%s/**" % folder)
    src = Path(tmp) / folder
    if src.exists():
        if dest.exists():
            shutil.copytree(str(src), str(dest), dirs_exist_ok=True)
        else:
            shutil.move(str(src), str(dest))
    else:
        print("[警告] 未找到 %s 内容" % folder)

# rmvpe.pt 位于仓库根目录，单独下载
rmvpe_dir = assets_dir / "rmvpe"
rmvpe_dir.mkdir(parents=True, exist_ok=True)
rmvpe_dst = rmvpe_dir / "rmvpe.pt"
if not rmvpe_dst.exists():
    print("下载 rmvpe.pt ...")
    hf_hub_download(repo_id=HF_REPO, filename="rmvpe.pt", local_dir=str(rmvpe_dir))
else:
    print("rmvpe.pt 已存在，跳过")

print("HuggingFace 包下载完成")


In [ ]:
# 校验模型与回退下载
repo = Path(WORK_DIR)
critical = {
    "hubert": [repo / "assets/hubert_base/pytorch_model.bin",
               repo / "assets/hubert_base/config.json",
               repo / "assets/hubert_base/preprocessor_config.json"],
    "rmvpe": [repo / "assets/rmvpe/rmvpe.pt"],
    "pretrain": [repo / "assets/pretrained/f0G40k.pth",
                 repo / "assets/pretrained_v2/f0G40k.pth",
                 repo / "assets/pretrained_v2/f0D40k.pth"],
}
for name, files in critical.items():
    missing = [f for f in files if not f.is_file()]
    print("%-10s %s" % (name, "OK" if not missing else "缺失: " + ", ".join(str(f) for f in missing)))
weights = sorted((repo / "assets/weights").glob("*.pth")) if (repo / "assets/weights").is_dir() else []
indices = sorted((repo / "assets/indices").glob("*.index")) if (repo / "assets/indices").is_dir() else []
print("示例权重 (assets/weights) : %d 个 .pth" % len(weights))
print("示例索引 (assets/indices) : %d 个 .index" % len(indices))
pymss_root = repo / "assets/pymss_weights"
PYMSS_NEEDED = [
    "dereverb_mel_band_roformer_less_aggressive_anvuew_sdr_18.8050.ckpt",
    "dereverb_mel_band_roformer_anvuew_sdr_19.1729.ckpt",
    "model_bs_roformer_ep_368_sdr_12.9628.ckpt",
    "model_bs_roformer_ep_317_sdr_12.9755.ckpt",
    "model_mel_band_roformer_karaoke_aufr33_viperx_sdr_10.1956.ckpt",
]
pymss_missing = [n for n in PYMSS_NEEDED if not (pymss_root / n).is_file()]
print("PyMSS(MSST) 分离权重     ：%d/5，缺失：%s" % (len(PYMSS_NEEDED) - len(pymss_missing), ", ".join(pymss_missing) if pymss_missing else "无"))
mute_dir = repo / "logs" / "mute"
print("logs/mute（训练静音样本）: %d 个文件" % (sum(1 for _ in mute_dir.iterdir()) if mute_dir.is_dir() else 0))
if FALLBACK_DOWNLOAD:
    missing_core = [f for files in critical.values() for f in files if not f.is_file()]
    if missing_core:
        from huggingface_hub import hf_hub_download
        print("开始下载缺失的核心模型...")
        for f in missing_core:
            try:
                rel = str(f.relative_to(Path(WORK_DIR) / "assets"))
                hf_hub_download(repo_id=HF_REPO, filename=rel, local_dir=str(f.parent))
                print("  已下载: %s" % f)
            except Exception as e:
                print("  下载失败 %s: %s" % (f, e))


In [ ]:
# 安装依赖（两阶段）
pv = sys.version_info[:2]
print("当前 Python：%d.%d" % pv)
DO_STAGE1 = pv == (3, 12)
if not DO_STAGE1:
    print("[警告] requirments_cu13_py312.txt 面向 Python 3.12；当前为 %d.%d。" % pv)
req = Path("requirments_cu13_py312.txt")
if not req.is_file():
    raise FileNotFoundError("未找到 %s，请先运行「克隆仓库」单元。" % req.name)
if DO_STAGE1:
    print("阶段 1/2：安装 torch / torchaudio（CUDA 13.0）...")
    extra = "--index-url %s" % TORCH_INDEX
    if PYPI_INDEX:
        extra += " --extra-index-url %s" % PYPI_INDEX
    pip_install("torch>=2.9.0+cu130,<2.15 torchaudio>=2.9.0+cu130,<2.15", extra)
print("阶段 2/2：安装项目依赖 %s ..." % req.name)
pip_install("-r " + req.name)
print("[完成] 依赖安装完成")


In [ ]:
# 环境自检
os.chdir(WORK_DIR)
import torch
print("torch     :", torch.__version__)
print("CUDA 可用 :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU       :", torch.cuda.get_device_name(0))
print("[完成] 核心库导入正常（部分告警可忽略）")


## 1. 语音转换 (Voice Conversion)

加载 RVC 模型并对输入音频进行音色转换。

In [ ]:
# ==================== 参数配置 ====================
# 语音转换参数（修改后运行此单元，再执行下方推理单元）
VC_MODEL_NAME = "kikiV1"        # 模型文件名（不含 .pth）
VC_INPUT_AUDIO = ""              # 输入音频路径
VC_OUTPUT_AUDIO = "output.wav"   # 输出音频路径
VC_SPEAKER_ID = 0                # 说话人 ID
VC_PITCH_SHIFT = 0               # 音调偏移（半音）
VC_F0_METHOD = "rmvpe"           # 音高提取方法: pm / rmvpe / fcpe
VC_INDEX_PATH = ""               # 索引文件路径（空=自动查找）
VC_INDEX_RATE = 0.75             # 索引权重 (0-1)
VC_RESAMPLE_SR = 0               # 重采样率 (0=不重采样)
VC_RMS_MIX_RATE = 1.0            # 响度混合率 (0-1)
VC_PROTECT = 0.33                # 保护参数 (0-0.5)
VC_OUTPUT_FORMAT = "wav"         # 输出格式: wav / flac / mp3 / m4a

print("VC 参数已设置")
print("  模型: %s" % VC_MODEL_NAME)
print("  输入: %s" % (VC_INPUT_AUDIO or "(未设置)"))
print("  输出: %s" % VC_OUTPUT_AUDIO)
print("  说话人ID: %d, 音调偏移: %d" % (VC_SPEAKER_ID, VC_PITCH_SHIFT))


In [ ]:
# 配置 RVC 环境变量
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("weight_root", str(Path(WORK_DIR) / "assets" / "weights"))
os.environ.setdefault("index_root", str(Path(WORK_DIR) / "logs"))
os.environ.setdefault("outside_index_root", str(Path(WORK_DIR) / "assets" / "indices"))
os.environ.setdefault("rmvpe_root", str(Path(WORK_DIR) / "assets" / "rmvpe"))
os.environ.setdefault("weight_pymss_root", str(Path(WORK_DIR) / "assets" / "pymss_weights"))
import numpy as np
import soundfile as sf
from pathlib import Path
print("项目目录：", WORK_DIR)
print("权重目录：", os.environ["weight_root"])
print("索引目录：", os.environ["index_root"])


In [ ]:
# 列出可用的模型
weights_dir = Path(os.environ["weight_root"])
if weights_dir.exists():
    models = sorted([p.name for p in weights_dir.glob("*.pth")])
    print("可用模型 (%d 个):" % len(models))
    for m in models[:20]:
        print("  -", m)
    if len(models) > 20:
        print("  ... 共 %d 个" % len(models))
else:
    print("权重目录不存在：", weights_dir)


In [ ]:
# 导入 RVC 模块并创建 VC 实例
from configs.config import Config
from i18n.i18n import I18nAuto
from infer.vc.modules import VC
i18n = I18nAuto()
config = Config()
print("设备：", config.device)
print("精度：", config.dtype)
vc = VC(config)
print("VC 实例已创建")


In [ ]:
# 加载模型
result = vc.get_vc(VC_MODEL_NAME)
print("模型加载完成")
print("目标采样率：", vc.tgt_sr)
print("版本：", vc.version)
print("是否使用音高：", vc.if_f0)


In [ ]:
# 获取模型信息
checkpoint = vc.cpt if hasattr(vc, 'cpt') and vc.cpt else None
if checkpoint:
    n_spk = checkpoint['config'][-3] if len(checkpoint['config']) > 3 else 0
    speaker_info = checkpoint.get('speaker_info', [])
    print("说话人数量：", n_spk)
    if speaker_info:
        print("说话人列表：")
        for s in speaker_info:
            print("  ID %d: %s" % (s["id"], s["name"]))
    else:
        print("单说话人模型")
else:
    print("未找到模型 checkpoint")


In [ ]:
# 语音转换函数
def voice_conversion(input_audio_path, output_path, speaker_id=0, pitch_shift=0,
    f0_method="rmvpe", index_path="", index_rate=0.75, resample_sr=0,
    rms_mix_rate=1.0, protect=0.33, output_format="wav"):
    if not Path(input_audio_path).exists():
        return False, "输入文件不存在: %s" % input_audio_path
    status, result = vc.vc_single(speaker_id=speaker_id, input_audio_path=input_audio_path,
        f0_up_key=pitch_shift, f0_method=f0_method, file_index=index_path,
        index_rate=index_rate, resample_sr=resample_sr, rms_mix_rate=rms_mix_rate, protect=protect)
    if not result or result[0] is None or result[1] is None:
        return False, status
    sample_rate, audio = result
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    if output_format in ("wav", "flac"):
        sf.write(str(output_path), audio, sample_rate)
    else:
        from io import BytesIO
        from infer.audio import wav2
        with BytesIO() as wav_buf:
            sf.write(wav_buf, audio, sample_rate, format="wav")
            wav_buf.seek(0)
            with open(output_path, "wb") as f:
                wav2(wav_buf, f, output_format)
    print(status)
    print("输出文件：", output_path)
    return True, str(output_path)


In [ ]:
# 执行语音转换
if VC_INPUT_AUDIO:
    success, result = voice_conversion(
        input_audio_path=VC_INPUT_AUDIO, output_path=VC_OUTPUT_AUDIO,
        speaker_id=VC_SPEAKER_ID, pitch_shift=VC_PITCH_SHIFT,
        f0_method=VC_F0_METHOD, index_path=VC_INDEX_PATH,
        index_rate=VC_INDEX_RATE, resample_sr=VC_RESAMPLE_SR,
        rms_mix_rate=VC_RMS_MIX_RATE, protect=VC_PROTECT,
        output_format=VC_OUTPUT_FORMAT)
    if success:
        print("转换成功！")
    else:
        print("转换失败：", result)
else:
    print("请设置 VC_INPUT_AUDIO 参数后运行")


## 2. 人声分离 (MSST)

使用 PyMSS 模型分离人声与伴奏。

In [ ]:
# ==================== 参数配置 ====================
# 人声分离参数（修改后运行此单元，再执行下方推理单元）
MSST_INPUT_AUDIO = ""  # 输入音频路径
MSST_OUTPUT_DIR = "separated"  # 输出目录
MSST_MODEL_NAME = "model_mel_band_roformer_karaoke_aufr33_viperx_sdr_10.1956.ckpt"
MSST_OVERLAP = 0.25  # 重叠比例
MSST_CHUNK_SIZE = 9.0  # 分块大小（秒）

print("MSST 参数已设置")
print("  输入: %s" % (MSST_INPUT_AUDIO or "(未设置)"))
print("  输出: %s" % MSST_OUTPUT_DIR)
print("  模型: %s" % MSST_MODEL_NAME)


In [ ]:
# 导入人声分离模块
from tools.pymss_webui import pymss_separate, stop_pymss_separation, PYMSS_MODEL_CHOICES, get_model_info
print("可用分离模型：")
for name in PYMSS_MODEL_CHOICES:
    info = get_model_info(name)
    print("  - %s: %s" % (name, info.get("label", "")))


In [ ]:
# 人声分离函数
def separate_audio(input_path, output_dir, model_name=None, device=None, overlap=0.25, chunk_size=9.0):
    if model_name is None:
        model_name = "model_mel_band_roformer_karaoke_aufr33_viperx_sdr_10.1956.ckpt"
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    result = pymss_separate(audio_path=str(input_path), model_name=model_name,
        output_dir=str(output_dir), overlap=overlap, chunk_size=chunk_size, device=device)
    if result:
        print("分离完成！")
        print("输出目录：", output_dir)
        return True, result
    return False, "分离失败"


In [ ]:
# 执行人声分离
if MSST_INPUT_AUDIO:
    success, result = separate_audio(
        input_path=MSST_INPUT_AUDIO, output_dir=MSST_OUTPUT_DIR,
        model_name=MSST_MODEL_NAME, overlap=MSST_OVERLAP, chunk_size=MSST_CHUNK_SIZE)
    if success:
        print("分离成功！")
    else:
        print("分离失败：", result)
else:
    print("请设置 MSST_INPUT_AUDIO 参数后运行")


## 3. 模型训练

对训练数据进行切片、特征提取并训练模型。

In [ ]:
# ==================== 参数配置 ====================
# 训练参数（修改后运行此单元，再执行下方训练单元）
TRAIN_AUDIO_DIR = "data/train"   # 训练音频目录
TRAIN_EXP_NAME = "my_voice"       # 实验目录名（logs/EXP_NAME）
TRAIN_SR = "40k"                   # 采样率: 32k / 40k / 48k
TRAIN_PER = 3.7                    # 每个片段的平均时长（秒）
TRAIN_THRESHOLD = -42              # 音量阈值
TRAIN_IF_F0 = True                 # 是否使用音高
TRAIN_SPEAKER_ID = 0               # 说话人 ID
TRAIN_TOTAL_EPOCH = 200            # 总训练轮数
TRAIN_SAVE_EPOCH = 50              # 每 N 轮保存一次检查点
TRAIN_BATCH_SIZE = 4               # 批大小
TRAIN_GPUS = "0"                   # 使用 GPU（如 0 或 0-1）
TRAIN_PRETRAINED_G = ""            # 生成器预训练模型路径（空=不使用）
TRAIN_PRETRAINED_D = ""            # 判别器预训练模型路径（空=不使用）
TRAIN_IF_SAVE_LATEST = False       # 仅保存最新检查点
TRAIN_IF_CACHE_GPU = False         # 将数据集缓存到 GPU 内存
TRAIN_IF_SAVE_EVERY_WEIGHTS = False # 每个检查点额外保存模型权重
TRAIN_F0_METHOD = "rmvpe"          # 音高提取方法: pm / rmvpe
TRAIN_N_CPU = 4                    # CPU 并行数（预处理和特征提取）

print("训练参数已设置")
print("  音频目录: %s" % TRAIN_AUDIO_DIR)
print("  实验目录: logs/%s" % TRAIN_EXP_NAME)
print("  采样率: %s, 使用音高: %s" % (TRAIN_SR, TRAIN_IF_F0))
print("  片段时长: %.1fs, 阈值: %d" % (TRAIN_PER, TRAIN_THRESHOLD))
print("  总轮数: %d, 保存间隔: %d, 批大小: %d" % (TRAIN_TOTAL_EPOCH, TRAIN_SAVE_EPOCH, TRAIN_BATCH_SIZE))
print("  GPU: %s, F0 方法: %s" % (TRAIN_GPUS, TRAIN_F0_METHOD))


In [ ]:
# 数据切分函数（调用 train/preprocess.py）
def run_preprocess(trainset_dir, exp_dir, sr, n_p, per=3.7):
    """运行 train/preprocess.py 进行数据切片。"""
    exp_dir_full = "logs/%s" % exp_dir
    os.makedirs(exp_dir_full, exist_ok=True)
    log_path = "%s/preprocess.log" % exp_dir_full
    with open(log_path, "w", encoding="utf8"):
        pass
    cmd = '"%s" train/preprocess.py "%s" %s %s "%s" %s %.1f' % (
        sys.executable, trainset_dir, sr, n_p, exp_dir_full,
        "True" if n_p == 1 else "False", per)
    print("[预处理] 命令:", cmd)
    with open(log_path, "a", encoding="utf8") as log_f:
        proc = subprocess.Popen(cmd, shell=True, stdout=log_f, stderr=subprocess.STDOUT,
                                cwd=str(PROJECT_ROOT), executable="/bin/bash")
        for line in proc.stdout:
            line = line.rstrip()
            if line:
                print("[预处理]", line)
        proc.wait()
    if proc.returncode != 0:
        raise RuntimeError("数据切分失败，返回码=%d" % proc.returncode)
    print("[预处理] 完成")
    return True


In [ ]:
# F0与HuBERT特征提取函数（调用 train/dataset/extract_f0.py）
def run_extract_f0(exp_dir, n_p, f0method, if_f0, gpus="0"):
    """运行 F0 和 HuBERT 特征提取。"""
    exp_dir_full = "logs/%s" % exp_dir
    log_path = "%s/extract_f0_feature.log" % exp_dir_full
    with open(log_path, "w", encoding="utf8"):
        pass
    if if_f0:
        is_cuda = torch.cuda.is_available() and gpus.strip()
        if is_cuda:
            gpu_list = gpus.split("-")
            processes = []
            for i, gpu_id in enumerate(gpu_list):
                cmd = '"%s" train/dataset/extract_f0.py cuda %s %s "%s" true %s' % (
                    sys.executable, len(gpu_list), i, exp_dir_full, f0method)
                print("[F0提取] GPU %s 命令:" % gpu_id, cmd)
                with open(log_path, "a", encoding="utf8") as log_f:
                    p = subprocess.Popen(cmd, shell=True, stdout=log_f, stderr=subprocess.STDOUT,
                                         cwd=str(PROJECT_ROOT), executable="/bin/bash")
                    processes.append(p)
            for p in processes:
                p.wait()
            if any(p.returncode != 0 for p in processes):
                raise RuntimeError("F0 提取失败")
        else:
            cmd = '"%s" train/dataset/extract_f0.py cpu "%s" %s %s' % (
                sys.executable, exp_dir_full, n_p, f0method)
            print("[F0提取] CPU 命令:", cmd)
            with open(log_path, "a", encoding="utf8") as log_f:
                proc = subprocess.Popen(cmd, shell=True, stdout=log_f, stderr=subprocess.STDOUT,
                                        cwd=str(PROJECT_ROOT), executable="/bin/bash")
                for line in proc.stdout:
                    line = line.rstrip()
                    if line:
                        print("[F0提取]", line)
                proc.wait()
            if proc.returncode != 0:
                raise RuntimeError("F0 提取失败")
    else:
        cmd = '"%s" train/dataset/extract_hubert_f0.py cpu "%s" %s' % (
            sys.executable, exp_dir_full, n_p)
        print("[特征提取] 命令:", cmd)
        with open(log_path, "a", encoding="utf8") as log_f:
            proc = subprocess.Popen(cmd, shell=True, stdout=log_f, stderr=subprocess.STDOUT,
                                    cwd=str(PROJECT_ROOT), executable="/bin/bash")
            for line in proc.stdout:
                line = line.rstrip()
                if line:
                    print("[特征提取]", line)
            proc.wait()
        if proc.returncode != 0:
            raise RuntimeError("特征提取失败")
    print("[F0/特征提取] 完成")
    return True


In [ ]:
# 模型训练函数（调用 train/train.py）
def run_train(exp_dir, sr, if_f0, spk_id, save_epoch, total_epoch, batch_size,
              pretrained_G="", pretrained_D="", gpus="0",
              if_save_latest=False, if_cache_gpu=False, if_save_every_weights=False, version="v2"):
    """运行 train/train.py 进行模型训练。"""
    exp_dir_full = "logs/%s" % exp_dir
    os.makedirs(exp_dir_full, exist_ok=True)
    # 生成 config.json
    from configs.config import Config
    cfg = Config()
    config_path = "v1/%s.json" % sr if version == "v1" or sr == "40k" else "v2/%s.json" % sr
    import copy
    config_data = copy.deepcopy(cfg.json_config[config_path])
    config_save_path = os.path.join(exp_dir_full, "config.json")
    with open(config_save_path, "w", encoding="utf8") as f:
        json.dump(config_data, f, ensure_ascii=False, indent=4, sort_keys=True)
        f.write("\n")
    # 生成 filelist.txt
    gt_wavs_dir = "%s/0_gt_wavs" % exp_dir_full
    feature_dir = "%s/3_feature256" % exp_dir_full if version == "v1" else "%s/3_feature768" % exp_dir_full
    if if_f0:
        f0_dir = "%s/2a_f0" % exp_dir_full
        f0nsf_dir = "%s/2b-f0nsf" % exp_dir_full
        names = (set([n.split(".")[0] for n in os.listdir(gt_wavs_dir)] if os.path.isdir(gt_wavs_dir) else [])
                 & set([n.split(".")[0] for n in os.listdir(feature_dir)] if os.path.isdir(feature_dir) else [])
                 & set([n.split(".")[0] for n in os.listdir(f0_dir)] if os.path.isdir(f0_dir) else [])
                 & set([n.split(".")[0] for n in os.listdir(f0nsf_dir)] if os.path.isdir(f0nsf_dir) else []))
    else:
        names = (set([n.split(".")[0] for n in os.listdir(gt_wavs_dir)] if os.path.isdir(gt_wavs_dir) else [])
                 & set([n.split(".")[0] for n in os.listdir(feature_dir)] if os.path.isdir(feature_dir) else []))
    if not names:
        raise RuntimeError("没有可用于训练的有效音频，请先完成数据切分和特征提取")
    lines = []
    for name in sorted(names):
        if if_f0:
            line = "%s/%s.wav|%s/%s.npy|%s/%s.wav.npy|%s/%s.wav.npy|%s" % (
                gt_wavs_dir.replace("\\", "/"), name, feature_dir.replace("\\", "/"), name,
                f0_dir.replace("\\", "/"), name, f0nsf_dir.replace("\\", "/"), name, spk_id)
        else:
            line = "%s/%s.wav|%s/%s.npy|%s" % (
                gt_wavs_dir.replace("\\", "/"), name, feature_dir.replace("\\", "/"), name, spk_id)
        lines.append(line)
    with open("%s/filelist.txt" % exp_dir_full, "w", encoding="utf8") as f:
        f.write("\n".join(lines))
    # 构建训练命令
    gpus_arg = "-g %s " % gpus if gpus else ""
    pg_arg = "-pg %s " % pretrained_G if pretrained_G else ""
    pd_arg = "-pd %s " % pretrained_D if pretrained_D else ""
    cmd = '"%s" train/train.py -e "%s" -sr %s -f0 %s -bs %s %s%s%s -te %s -se %s -l %s -c %s -sw %s -v %s' % (
        sys.executable, exp_dir, sr, 1 if if_f0 else 0, batch_size,
        gpus_arg, pg_arg, pd_arg,
        total_epoch, save_epoch,
        1 if if_save_latest else 0, 1 if if_cache_gpu else 0,
        1 if if_save_every_weights else 0, version)
    log_path = "%s/train.log" % exp_dir_full
    print("[训练] 命令:", cmd)
    with open(log_path, "w", encoding="utf8"):
        pass
    env = os.environ.copy()
    env["RVC_CUDA_GRAPH"] = "0"
    with open(log_path, "a", encoding="utf8") as log_f:
        proc = subprocess.Popen(cmd, shell=True, stdout=log_f, stderr=subprocess.STDOUT,
                                cwd=str(PROJECT_ROOT), executable="/bin/bash", env=env)
        for line in proc.stdout:
            line = line.rstrip()
            if line:
                print("[训练]", line)
        proc.wait()
    if proc.returncode != 0:
        raise RuntimeError("模型训练失败，返回码=%d" % proc.returncode)
    print("[训练] 完成")
    return True


In [ ]:
# 一键训练函数：预处理 + F0提取 + 模型训练
def train_oneclick(
    trainset_dir,
    exp_dir=None,
    sr=None,
    if_f0=None,
    spk_id=None,
    n_p=None,
    f0method=None,
    per=None,
    save_epoch=None,
    total_epoch=None,
    batch_size=None,
    pretrained_G="",
    pretrained_D="",
    gpus=None,
    if_cache_gpu=False,
    if_save_every_weights=False,
    version=None,
):
    """
    一键训练：数据切分 → F0/特征提取 → 模型训练 → 索引训练
    省略参数时使用全局 TRAIN_* 变量。
    """
    exp_dir  = exp_dir  or TRAIN_EXP_NAME
    sr       = sr       or TRAIN_SR
    if_f0    = if_f0   if if_f0 is not None else TRAIN_IF_F0
    spk_id   = spk_id   if spk_id   is not None else TRAIN_SPEAKER_ID
    n_p      = n_p      or TRAIN_N_CPU
    f0method = f0method or TRAIN_F0_METHOD
    save_epoch = save_epoch or TRAIN_SAVE_EPOCH
    total_epoch = total_epoch or TRAIN_TOTAL_EPOCH
    batch_size = batch_size or TRAIN_BATCH_SIZE
    gpus     = gpus     or TRAIN_GPUS
    version  = version  or "v2"
    per      = per      or TRAIN_PER
    steps = []
    try:
        # Step 1: 数据切分
        print("=== Step 1/4: 数据切分 ===")
        run_preprocess(trainset_dir, exp_dir, sr, n_p, per)
        steps.append("数据切分")
        # Step 2: F0与特征提取
        print("=== Step 2/4: F0与HuBERT特征提取 ===")
        run_extract_f0(exp_dir, n_p, f0method, if_f0, gpus)
        steps.append("F0/特征提取")
        # Step 3: 模型训练
        print("=== Step 3/4: 模型训练 ===")
        run_train(
            exp_dir, sr, if_f0, spk_id, save_epoch, total_epoch, batch_size,
            pretrained_G=pretrained_G, pretrained_D=pretrained_D, gpus=gpus,
            if_cache_gpu=if_cache_gpu, if_save_every_weights=if_save_every_weights,
            version=version)
        steps.append("模型训练")
        # Step 4: 索引训练
        print("=== Step 4/4: 索引训练 ===")
        idx_ok = train_index_for_model(exp_dir, version=version)
        if idx_ok:
            steps.append("索引训练")
        print("\n=== 训练全流程完成 ===")
        print("已完成步骤:", " -> ".join(steps))
        return True, steps
    except Exception as e:
        print("\n[错误] 训练失败:", e)
        import traceback
        traceback.print_exc()
        return False, steps


In [ ]:
# 执行一键训练
# 修改参数后运行此单元即可自动完成全部训练流程
if TRAIN_AUDIO_DIR and Path(TRAIN_AUDIO_DIR).exists():
    success, steps = train_oneclick(
        trainset_dir=TRAIN_AUDIO_DIR,
        per=TRAIN_PER,
        exp_dir=TRAIN_EXP_NAME,
        sr=TRAIN_SR,
        if_f0=TRAIN_IF_F0,
        spk_id=TRAIN_SPEAKER_ID,
        n_p=TRAIN_N_CPU,
        f0method=TRAIN_F0_METHOD,
        save_epoch=TRAIN_SAVE_EPOCH,
        total_epoch=TRAIN_TOTAL_EPOCH,
        batch_size=TRAIN_BATCH_SIZE,
        pretrained_G=TRAIN_PRETRAINED_G,
        pretrained_D=TRAIN_PRETRAINED_D,
        gpus=TRAIN_GPUS,
        if_cache_gpu=TRAIN_IF_CACHE_GPU,
        if_save_every_weights=TRAIN_IF_SAVE_EVERY_WEIGHTS,
        version="v2" if TRAIN_SR in ("48k",) or True else "v1",
    )
    if success:
        print("一键训练全部完成！")
else:
    print("请确保 TRAIN_AUDIO_DIR 目录存在后再运行")


## 4. 索引训练

使用 FAISS 训练特征索引。

In [ ]:
# ==================== 参数配置 ====================
# 索引训练参数（修改后运行此单元，再执行下方训练单元）
INDEX_EXP_NAME = "my_voice"  # 实验名称（需与预处理目录一致）
INDEX_VERSION = "v2"  # 模型版本: v1 / v2
INDEX_N_CPU = 4  # CPU 线程数

print("索引训练参数已设置")
print("  实验名称: %s" % INDEX_EXP_NAME)
print("  版本: %s, CPU线程: %d" % (INDEX_VERSION, INDEX_N_CPU))


In [ ]:
# 导入索引训练模块
import faiss
import traceback
print("索引训练模块已导入")


In [ ]:
# 索引训练函数
def train_index_for_model(exp_name, version='v2', n_cpu=4, speaker_id=None):
    exp_dir = Path("logs") / exp_name
    feature_dir = exp_dir / ("3_feature256" if version == "v1" else "3_feature768")
    if not feature_dir.exists():
        print("特征目录不存在：%s" % feature_dir)
        return False
    try:
        feature_files = sorted(feature_dir.glob("*.npy"))
        if not feature_files:
            print("未找到特征文件")
            return False
        print("加载 %d 个特征文件..." % len(feature_files))
        features = [np.load(f) for f in feature_files]
        all_features = np.concatenate(features, axis=0)
        print("特征维度：", all_features.shape)
        dimension = all_features.shape[1]
        index = faiss.IndexFlatL2(dimension)
        index.add(all_features)
        index_path = exp_dir / ("%s_trained.index" % exp_name)
        faiss.write_index(index, str(index_path))
        outside_dir = Path(os.environ.get("outside_index_root", str(Path(WORK_DIR) / "assets" / "indices")))
        outside_dir.mkdir(parents=True, exist_ok=True)
        link_path = outside_dir / index_path.name
        if not link_path.exists():
            os.symlink(str(index_path), str(link_path))
        print("索引训练完成！")
        print("索引文件：", index_path)
        return True
    except Exception as e:
        print("索引训练失败：", e)
        traceback.print_exc()
        return False


In [ ]:
# 执行索引训练
if INDEX_EXP_NAME:
    success = train_index_for_model(exp_name=INDEX_EXP_NAME, version=INDEX_VERSION, n_cpu=INDEX_N_CPU)
    if success:
        print("索引训练成功！")
else:
    print("请设置 INDEX_EXP_NAME 并确保预处理已完成")


## 5. 模型导出

保存训练好的模型。

In [ ]:
# ==================== 参数配置 ====================
# 模型导出参数（修改后运行此单元，再执行下方导出单元）
EXPORT_CHECKPOINT_PATH = "logs/my_voice/model.pth"  # 检查点文件路径
EXPORT_OUTPUT_NAME = "my_voice"  # 输出模型名称（不含扩展名）
EXPORT_EPOCH = 1000  # 训练轮数
EXPORT_SR = 40000  # 采样率
EXPORT_IF_F0 = True  # 是否使用音高
EXPORT_VERSION = "v2"  # 模型版本

print("模型导出参数已设置")
print("  检查点: %s" % EXPORT_CHECKPOINT_PATH)
print("  输出: %s (%sepoch, %s, %s)" % (EXPORT_OUTPUT_NAME, EXPORT_EPOCH, EXPORT_IF_F0, EXPORT_VERSION))


In [ ]:
# 导入模型导出模块
from train.process_ckpt import savee, show_info
print("模型导出模块已导入")


In [ ]:
# 查看已有模型信息
def show_model_info(model_path):
    path = Path(model_path)
    if not path.exists():
        print("模型不存在：%s" % path)
        return
    info = show_info(str(path))
    print(info)
    try:
        ckpt = torch.load(str(path), map_location="cpu")
        if 'weight' in ckpt:
            weight = ckpt['weight']
            print("\n模型参数维度：")
            for k, v in weight.items():
                if hasattr(v, 'shape'):
                    print("  %s: %s" % (k, str(v.shape)))
    except Exception as e:
        print("读取模型失败：", e)


In [ ]:
# 导出训练好的模型
def export_model(checkpoint_path, output_name, epoch, sr=40000, if_f0=True, version='v2'):
    try:
        checkpoint = torch.load(str(checkpoint_path), map_location="cpu")
        class Hps:
            def __init__(self):
                self.data = type('D', (), {'filter_length': 2048, 'sampling_rate': sr})()
                self.model = type('M', (), {
                    'inter_channels': 192, 'hidden_channels': 192, 'filter_channels': 768,
                    'n_heads': 2, 'n_layers': 6, 'kernel_size': 3, 'p_dropout': 0,
                    'resblock': "1", 'resblock_kernel_sizes': [3, 7, 11],
                    'resblock_dilation_sizes': [[1, 3, 5], [1, 3, 5], [1, 3, 5]],
                    'upsample_rates': [10, 10], 'upsample_initial_channel': 512,
                    'upsample_kernel_sizes': [20, 20], 'spk_embed_dim': 108, 'gin_channels': 256,
                })()
                self.spk_embed_dim = 108
        hps = Hps()
        result = savee(checkpoint, sr, if_f0, output_name, epoch, version, hps)
        print("导出结果：", result)
        output_path = Path(os.environ.get("weight_root", str(Path(WORK_DIR) / "assets" / "weights"))) / (output_name + ".pth")
        if output_path.exists():
            show_model_info(str(output_path))
        return True
    except Exception as e:
        print("导出失败：", e)
        traceback.print_exc()
        return False


In [ ]:
# 执行模型导出
if EXPORT_CHECKPOINT_PATH and Path(EXPORT_CHECKPOINT_PATH).exists():
    success = export_model(
        checkpoint_path=EXPORT_CHECKPOINT_PATH, output_name=EXPORT_OUTPUT_NAME,
        epoch=EXPORT_EPOCH, sr=EXPORT_SR, if_f0=EXPORT_IF_F0, version=EXPORT_VERSION)
    if success:
        print("模型导出成功！")
else:
    print("请设置 EXPORT_CHECKPOINT_PATH 并确保文件存在")


In [ ]:
# 查看现有模型
weights_dir = Path(os.environ["weight_root"])
if weights_dir.exists():
    for p in sorted(weights_dir.glob("*.pth")):
        print("\n" + "="*50)
        show_model_info(str(p))
else:
    print("权重目录不存在")


## 6. 批量处理

对多个音频文件进行批量语音转换。

In [ ]:
# ==================== 参数配置 ====================
# 批量处理参数（修改后运行此单元，再执行下方推理单元）
BATCH_INPUT_DIR = "input_audios"  # 输入目录
BATCH_OUTPUT_DIR = "output_audios"  # 输出目录
BATCH_SPEAKER_ID = 0  # 说话人 ID
BATCH_PITCH_SHIFT = 0  # 音调偏移（半音）
BATCH_INDEX_RATE = 0.75  # 索引权重
BATCH_RECURSIVE = False  # 是否递归子目录

print("批量处理参数已设置")
print("  输入: %s" % BATCH_INPUT_DIR)
print("  输出: %s" % BATCH_OUTPUT_DIR)
print("  说话人ID: %d, 音调偏移: %d, 递归: %s" % (BATCH_SPEAKER_ID, BATCH_PITCH_SHIFT, BATCH_RECURSIVE))


In [ ]:
# 批量转换函数
def batch_convert(input_dir, output_dir, speaker_id=0, pitch_shift=0, index_rate=0.75, recursive=False):
    input_dir = Path(input_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    patterns = ["*.wav", "*.mp3", "*.flac", "*.m4a"]
    audio_files = []
    for pat in patterns:
        audio_files.extend(input_dir.rglob(pat) if recursive else input_dir.glob(pat))
    if not audio_files:
        print("未找到音频文件")
        return 0
    print("找到 %d 个音频文件" % len(audio_files))
    index_path = ""
    indices_dir = Path(os.environ.get("outside_index_root", str(Path(WORK_DIR) / "assets" / "indices")))
    if indices_dir.exists():
        for idx in sorted(indices_dir.glob("*.index")):
            index_path = str(idx)
            break
    success_count = 0
    for i, audio_file in enumerate(audio_files):
        print("\n[%d/%d] 处理: %s" % (i+1, len(audio_files), audio_file.name))
        output_file = output_dir / (audio_file.stem + ".wav")
        success, result = voice_conversion(
            input_audio_path=str(audio_file), output_path=str(output_file),
            speaker_id=speaker_id, pitch_shift=pitch_shift,
            index_path=index_path, index_rate=index_rate)
        if success:
            success_count += 1
        else:
            print("失败：", result)
    print("\n批量转换完成: %d/%d" % (success_count, len(audio_files)))
    return success_count


In [ ]:
# 执行批量转换
if BATCH_INPUT_DIR and Path(BATCH_INPUT_DIR).exists():
    count = batch_convert(
        input_dir=BATCH_INPUT_DIR, output_dir=BATCH_OUTPUT_DIR,
        speaker_id=BATCH_SPEAKER_ID, pitch_shift=BATCH_PITCH_SHIFT,
        index_rate=BATCH_INDEX_RATE, recursive=BATCH_RECURSIVE)
    print("完成！")
else:
    print("请设置 BATCH_INPUT_DIR 并确保目录存在")


## 使用提示

1. **环境配置**：按顺序运行 `helpers` → `config` → `env-check` → `system-tools` → `clone-repo` → `download-pkg` → `verify-models` → `install-deps` → `self-check`

2. **语音转换**：先运行 `setup-rvc` → `init-vc` → `load-model`，再修改 `vc-params` 参数后运行 `vc-demo`

3. **人声分离**：运行 `msst-import` → `msst-function`，再修改 `msst-params` 参数后运行 `msst-demo`

4. **一键训练**：修改 `train-params` 参数后，运行 `train-demo` 自动执行全部训练步骤

5. **模型导出**：运行 `export-imports`，再修改 `export-params` 参数后运行 `export-demo`

6. **批量处理**：运行 `batch-function`，再修改 `batch-params` 参数后运行 `batch-demo`

注意事项：
- 确保已下载模型权重到 `assets/weights/`
- 确保已训练索引文件到 `assets/indices/`
- 首次运行可能需要下载依赖
- 安装包下载约 7.8 GB，需要足够磁盘空间
- 训练需要充足 GPU 显存（建议 ≥ 8GB）
